In [1]:
import spacy
spacy.prefer_gpu()
nlp = spacy.load("da_core_news_trf")


/Users/rakul/Desktop/cs/5sem/msc/cph-menu-finder/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [2]:
import pandas as pd
import re
from config import WOLT_DATA_DIR

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 60)

In [3]:
AREA = "vesterbro"

venues = pd.read_csv(WOLT_DATA_DIR / AREA / "venues.csv")
items_raw = pd.read_csv(WOLT_DATA_DIR / AREA / "items.csv")

## Lemma

In [ ]:
# built by build_wolt_data_model.py, verified with annotate_ingredients.py
# venues 1-N items N-M ingredients (via item_ingredients) N-1 ingredient_verified
item_ingredients = pd.read_csv(WOLT_DATA_DIR / AREA / "item_ingredients.csv")
ingredients = pd.read_csv(WOLT_DATA_DIR / AREA / "ingredients.csv")   # is_ingredient: 1, 0 or NaN (not reviewed yet)

# one row per (venue, item, raw ingredient), with its verified ingredient
venue_ingredients = (
    item_ingredients
    .merge(items_raw[["item_id", "wolt_id", "item"]], on="item_id")
    .merge(venues[["wolt_id", "name"]], on="wolt_id")
    .merge(ingredients, on="ingredient_raw")
)
venue_ingredients = venue_ingredients[venue_ingredients.is_ingredient == 1]   # only reviewed, real ingredients

In [25]:
from googletrans import Translator

translator = Translator()

langs = await translator.detect(items_raw.description)


In [29]:
unique_descs = [ d for d in items_raw.description.unique() ]
detections = await translator.detect(unique_descs)

lang_by_desc = { d: det.lang for d, det in zip(unique_descs, detections) }
confidence_by_desc = { d: det.confidence for d, det in zip(unique_descs, detections) }

items_raw["lang"] = items_raw.description.map(lang_by_desc)
items_raw["lang_confidence"] = items_raw.description.map(confidence_by_desc)
items_raw

,wolt_id,venue,category,item,description,price_dkk,description_lemmatized,lang,lang_confidence
0,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Burger,Klassisk okseburger,"sesambolle med fyld af oksebøf, dressing, ketchup, icebe...",88.0,"[sesambolle, fyld, oksebøf, dressing, ketchup, icebergsa...",da,0.975193
1,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Hovedretter,En halv grillkylling,,108.0,[],en,0.000000
2,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Burger,Maxi Cheeseburger,"briochebolle med fyld af ca. 300 grams oksebøf, ost, sal...",128.0,"[briochebolle, fyld, cirka, 300, gram, oksebøf, ost, sal...",da,0.988594
3,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Pizzaer,Gorgonzola,"tomatsauce, ost, gorgonzola, champignon, løg og rucola.",115.0,"[tomatsauce, ost, gorgonzola, champignon, løg, rucola]",da,0.982142
4,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Pizzaer,Caprese,"tomatsauce, ost, kødstrimler, paprika og bearnaisesauce.",115.0,"[tomatsauce, ost, kødstrimler, paprika, bearnaisesauce]",da,1.000000
...,...,...,...,...,...,...,...,...,...
9962,5c7796f8e50c02000bbc7523,Zócalo Tivoli Food Hall,Quesadillas,Mini Corn Quesadilla,"corn tortilla, ost, carnitas pork eller kylling, salsa m...",65.0,"[corn, tortilla, ost, carnitas, pork, kylling, salsa, ma...",en,1.000000
9963,5c7796f8e50c02000bbc7523,Zócalo Tivoli Food Hall,Burritos,Burrito - Block Party BBQ 🌶️,"hvede tortilla, ris, raw slaw, sorte bønner, dit valg af...",145.0,"[hvede, tortilla, ris, raw, slaw, sort, bønn, din, valg,...",en,1.000000
9964,5c7796f8e50c02000bbc7523,Zócalo Tivoli Food Hall,Børnemenu,Kids Bowl,"ris, romaine, barbacoa beef/grillet kylling/veggie strip...",89.0,"[ris, romaine, barbacoa, beef, grillet, kylling, veggie,...",en,1.000000
9965,5c7796f8e50c02000bbc7523,Zócalo Tivoli Food Hall,Dips,BBQ Sauce,.,10.0,[],en,1.000000


In [13]:
for d in items_raw.lemmas:
    print(d)

['sesambolle', 'fyld', 'oksebøf', 'dressing', 'ketchup', 'icebergsalat', 'løgring', 'tomat', 'agurk']
[]
['briochebolle', 'fyld', 'cirka', '300', 'gram', 'oksebøf', 'ost', 'salat', 'tomat', 'løg', 'agurk']
['tomatsauce', 'ost', 'gorgonzola', 'champignon', 'løg', 'rucola']
['tomatsauce', 'ost', 'kødstrimler', 'paprika', 'bearnaisesauce']
['tomatsauce', 'ost', 'kylling', 'pepperoni', 'paprika']
['tomatsauce', 'ost', 'spinat', 'oliven', 'gorgonzola', 'champignon', 'hvidløg']
['pommes', 'frites', 'salat', 'remoulade']
['tomatsauce', 'ost', 'kebab', 'kylling', 'pepperoni', 'kalkunskinke', 'salat', 'dressing']
['oksekødsret', 'pommes', 'frites', 'hummus', 'salat', 'brød']
[]
['tyrkisk', 'brød', 'fyld', 'valgfri', 'protein', 'salat', 'tomat', 'agurk', 'creme', 'fraiche']
['box', 'fyld', 'kebab', 'salat', 'pommes', 'frites', 'dressing']
['tomatsauce', 'ost', 'kalkunskinke']
['pommes', 'frites', 'salat', 'dressing', 'remoulade']
[]
['tomatsauce', 'ost', 'kebab', 'champignon']
['græsk', 'bøf', '

In [9]:
items_raw

,wolt_id,venue,category,item,description,price_dkk,lemmas
0,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Burger,Klassisk okseburger,"sesambolle med fyld af oksebøf, dressing, ketchup, icebe...",88.0,"[sesambolle, fyld, oksebøf, dressing, ketchup, icebergsa..."
1,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Hovedretter,En halv grillkylling,,108.0,[]
2,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Burger,Maxi Cheeseburger,"briochebolle med fyld af ca. 300 grams oksebøf, ost, sal...",128.0,"[briochebolle, fyld, cirka, 300, gram, oksebøf, ost, sal..."
3,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Pizzaer,Gorgonzola,"tomatsauce, ost, gorgonzola, champignon, løg og rucola.",115.0,"[tomatsauce, ost, gorgonzola, champignon, løg, rucola]"
4,6a71e83b2c4b0028d09cf99e,1001 Nat Pizza,Pizzaer,Caprese,"tomatsauce, ost, kødstrimler, paprika og bearnaisesauce.",115.0,"[tomatsauce, ost, kødstrimler, paprika, bearnaisesauce]"
...,...,...,...,...,...,...,...
9962,5c7796f8e50c02000bbc7523,Zócalo Tivoli Food Hall,Quesadillas,Mini Corn Quesadilla,"corn tortilla, ost, carnitas pork eller kylling, salsa m...",65.0,"[corn, tortilla, ost, carnitas, pork, kylling, salsa, ma..."
9963,5c7796f8e50c02000bbc7523,Zócalo Tivoli Food Hall,Burritos,Burrito - Block Party BBQ 🌶️,"hvede tortilla, ris, raw slaw, sorte bønner, dit valg af...",145.0,"[hvede, tortilla, ris, raw, slaw, sort, bønn, din, valg,..."
9964,5c7796f8e50c02000bbc7523,Zócalo Tivoli Food Hall,Børnemenu,Kids Bowl,"ris, romaine, barbacoa beef/grillet kylling/veggie strip...",89.0,"[ris, romaine, barbacoa, beef, grillet, kylling, veggie,..."
9965,5c7796f8e50c02000bbc7523,Zócalo Tivoli Food Hall,Dips,BBQ Sauce,.,10.0,[]


In [17]:
import json

with open(WOLT_DATA_DIR / AREA / "description_ingredients.json") as f:
    desc_to_ingr = json.load(f)

items_raw["ingredients"] = items_raw.description.map(lambda d: desc_to_ingr.get(d, []))
for i in items_raw.ingredients:
    print(i)

['sesambolle', 'oksekød', 'dressing', 'ketchup', 'icebergsalat', 'løgring', 'tomat', 'agurk']
[]
['briochebolle', 'oksekød', 'ost', 'salat', 'tomat', 'løg', 'agurk']
['tomatsauce', 'ost', 'gorgonzola', 'champignon', 'løg', 'rucola']
['tomatsauce', 'ost', 'kødstrimler', 'peberfrugt', 'bearnaise']
['tomatsauce', 'ost', 'kylling', 'pepperoni', 'peberfrugt']
['tomatsauce', 'ost', 'spinat', 'oliven', 'gorgonzola', 'champignon', 'hvidløg']
['pommes frites', 'salat', 'remoulade']
['tomatsauce', 'ost', 'kebab', 'kylling', 'pepperoni', 'kalkunskinke', 'salat', 'dressing']
['oksekød', 'pommes frites', 'hummus', 'salat', 'brød']
[]
['tyrkisk brød', 'salat', 'tomat', 'agurk', 'creme fraiche']
['kebab', 'salat', 'pommes frites', 'dressing']
['tomatsauce', 'ost', 'kalkunskinke']
['pommes frites', 'salat', 'dressing', 'remoulade']
[]
['tomatsauce', 'ost', 'kebab', 'champignon']
['oksekød', 'løg', 'pommes frites', 'jalapeño', 'salat', 'bearnaise']
['tomatsauce', 'ost', 'reje', 'tun', 'peberfrugt', 'lø